# BERT Manual Implementation


**Pipeline:**
```
Input Text → Tokenize → Input Embedding + Positional Encoding
           → [Nx] Multi-Head Self-Attention → Add & Norm → Feed Forward → Add & Norm
           → Linear → Softmax → Output
```

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ─── Config ────────────────────────────────────────────────────────────────────
SENTENCE     = "I love deep learning with BERT model"
D_MODEL      = 8        # embedding dimension (nhỏ để dễ debug)
NUM_HEADS    = 2        # số heads trong Multi-Head Attention
D_FF         = 16       # Feed Forward hidden size
NUM_LAYERS   = 2        # số Encoder Block (Nx)
DROPOUT      = 0.0      # tắt dropout khi debug

print("=" * 60)
print("CONFIG")
print("=" * 60)
print(f"  Sentence   : {SENTENCE!r}")
print(f"  d_model    : {D_MODEL}")
print(f"  num_heads  : {NUM_HEADS}")
print(f"  d_ff       : {D_FF}")
print(f"  num_layers : {NUM_LAYERS}")
print(f"  d_head     : {D_MODEL // NUM_HEADS}  (d_model / num_heads)")

CONFIG
  Sentence   : 'I love deep learning with BERT model'
  d_model    : 8
  num_heads  : 2
  d_ff       : 16
  num_layers : 2
  d_head     : 4  (d_model / num_heads)


---
## Tokenization & Vocabulary
Chuyển câu văn thành danh sách token IDs.

In [13]:
words = SENTENCE.lower().split()
vocab = ["[PAD]", "[CLS]", "[SEP]"] + sorted(set(words))  # BERT luôn có special tokens
vocab_size = len(vocab)

stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for i, w in enumerate(vocab)}

# BERT thêm [CLS] đầu và [SEP] cuối
tokens     = ["[CLS]"] + words + ["[SEP]"]
token_ids  = [stoi[t] for t in tokens]
seq_len    = len(tokens)
input_ids  = torch.tensor(token_ids, dtype=torch.long)

print("=" * 60)
print("STEP 1: TOKENIZATION")
print("=" * 60)
print(f"  Câu gốc       : {SENTENCE!r}")
print(f"  Tokens        : {tokens}")
print(f"  Token IDs     : {token_ids}")
print(f"  Input tensor  : shape={list(input_ids.shape)}, dtype={input_ids.dtype}")
print(f"  Vocab size    : {vocab_size}")
print(f"  Seq length    : {seq_len}")
print()
print("  Vocab mapping:")
for k, v in stoi.items():
    print(f"    '{k}' → {v}")

STEP 1: TOKENIZATION
  Câu gốc       : 'I love deep learning with BERT model'
  Tokens        : ['[CLS]', 'i', 'love', 'deep', 'learning', 'with', 'bert', 'model', '[SEP]']
  Token IDs     : [1, 5, 7, 4, 6, 9, 3, 8, 2]
  Input tensor  : shape=[9], dtype=torch.int64
  Vocab size    : 10
  Seq length    : 9

  Vocab mapping:
    '[PAD]' → 0
    '[CLS]' → 1
    '[SEP]' → 2
    'bert' → 3
    'deep' → 4
    'i' → 5
    'learning' → 6
    'love' → 7
    'model' → 8
    'with' → 9


---
## Input Embedding + Positional Encoding
- **Word Embedding**: map token ID → vector d_model chiều
- **Positional Encoding**: thêm thông tin vị trí bằng sin/cos
- **Cộng lại** → vector chứa cả nghĩa + vị trí

In [14]:
torch.manual_seed(42)

# ── Word Embedding ──────────────────────────────────────────────────────────────
E = torch.randn(vocab_size, D_MODEL)           # [vocab_size, d_model]
word_emb = E[input_ids]                        # [seq_len, d_model]
word_emb_scaled = word_emb * np.sqrt(D_MODEL)  # scale như Transformer gốc

print("=" * 60)
print("STEP 2a: WORD EMBEDDING")
print("=" * 60)
print(f"  Embedding matrix E     : shape={list(E.shape)}")
print(f"  word_emb (raw)         : shape={list(word_emb.shape)}")
print(f"  word_emb (scaled ×√{D_MODEL}) : shape={list(word_emb_scaled.shape)}")
print()
for i, (tok, vec) in enumerate(zip(tokens, word_emb_scaled)):
    print(f"  [{i}] '{tok:10s}' → {vec.detach().numpy().round(3)}")

# ── Positional Encoding ─────────────────────────────────────────────────────────
pe       = torch.zeros(seq_len, D_MODEL)
position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)  # [seq_len, 1]
div_term = torch.exp(torch.arange(0, D_MODEL, 2).float() * -(np.log(10000.0) / D_MODEL))

pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)

print()
print("=" * 60)
print("STEP 2b: POSITIONAL ENCODING")
print("=" * 60)
print(f"  position shape : {list(position.shape)}")
print(f"  div_term       : {div_term.numpy().round(4)}  ← {len(div_term)} tần số cho {D_MODEL} chiều")
print(f"  pe shape       : {list(pe.shape)}")
print()
for i, (tok, vec) in enumerate(zip(tokens, pe)):
    print(f"  pos[{i}] '{tok:10s}' → {vec.detach().numpy().round(3)}")

# ── Cộng lại ────────────────────────────────────────────────────────────────────
x = word_emb_scaled + pe   # [seq_len, d_model]
x = x.unsqueeze(0)        # [1, seq_len, d_model] — thêm batch dim

print()
print("=" * 60)
print("STEP 2c: word_emb_scaled + pe  →  x")
print("=" * 60)
print(f"  x shape : {list(x.shape)}  [batch=1, seq_len={seq_len}, d_model={D_MODEL}]")
print()
for i, tok in enumerate(tokens):
    print(f"  x[0, {i}] '{tok:10s}' → {x[0, i].detach().numpy().round(3)}")

STEP 2a: WORD EMBEDDING
  Embedding matrix E     : shape=[10, 8]
  word_emb (raw)         : shape=[9, 8]
  word_emb (scaled ×√8) : shape=[9, 8]

  [0] '[CLS]     ' → [-2.127  4.663 -1.11  -3.97  -2.059 -1.582 -2.175  2.157]
  [1] 'i         ' → [-4.405  2.816 -2.488 -1.7   -3.604  6.004 -3.492 -1.38 ]
  [2] 'love      ' → [-3.969  0.102 -0.18   1.911 -0.277  5.217 -3.35   3.913]
  [3] 'deep      ' → [-3.916 -2.464 -0.632  4.857  0.902 -1.201  0.865 -2.191]
  [4] 'learning  ' → [-2.585 -1.861  0.221  1.487 -1.38   3.37  -2.302 -2.082]
  [5] 'with      ' → [-0.487  1.481  0.16   1.206  1.626 -1.815 -6.241 -2.124]
  [6] 'bert      ' → [ 3.618  3.667  1.727  3.775 -0.655  0.118 -0.712  2.432]
  [7] 'model     ' → [ 4.087  2.422  6.274  1.48   0.98  -0.558 -2.983  3.615]
  [8] '[SEP]     ' → [ 4.645 -0.451 -1.407  1.243 -2.144  3.05   2.265  4.754]

STEP 2b: POSITIONAL ENCODING
  position shape : [9, 1]
  div_term       : [1.    0.1   0.01  0.001]  ← 4 tần số cho 8 chiều
  pe shape       : 

---
## Self-Attention (1 head)
Tính toán: Q, K, V → Attention Score → Softmax → Weighted Sum

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [15]:
# ── DEMO: Tại sao Q[9×4] @ K.T[4×9] = scores[9×9] ─────────────────────────────
# Giải thích bằng số nhỏ d_k=2, seq_len=4 từ

demo_tokens = ["[CLS]", "I", "love", "deep"]
demo_seq    = len(demo_tokens)
demo_dk     = 2   # chỉ 2 chiều cho dễ nhìn

# Giả sử Q, K đơn giản
Q_demo = torch.tensor([[1., 0.],   # [CLS]  hỏi theo hướng [1,0]
                        [1., 0.],   # "I"    hỏi theo hướng [1,0]
                        [0., 1.],   # "love" hỏi theo hướng [0,1]
                        [1., 1.]],  # "deep" hỏi theo hướng [1,1]
                       dtype=torch.float)

K_demo = torch.tensor([[1., 0.],   # [CLS]  key: liên quan hướng [1,0]
                        [1., 0.],   # "I"    key: liên quan hướng [1,0]
                        [0., 1.],   # "love" key: liên quan hướng [0,1]
                        [1., 1.]],  # "deep" key: liên quan hướng [1,1]
                       dtype=torch.float)

scores_demo = Q_demo @ K_demo.T / np.sqrt(demo_dk)  # [4×4]
attn_demo   = F.softmax(scores_demo, dim=-1)

print("=" * 70)
print("DEMO: Cách Q @ K.T tạo ra ma trận 'từ nào attend từ nào'")
print("=" * 70)
print(f"  Q_demo shape : {list(Q_demo.shape)}  ← mỗi hàng = vector Query của 1 từ")
print(f"  K_demo shape : {list(K_demo.shape)}  ← mỗi hàng = vector Key của 1 từ")
print()
print(f"  Q_demo @ K_demo.T → scores shape : {list(scores_demo.shape)}")
print()
print("  scores[i][j] = Q_i · K_j = dot product = 'mức độ match' của từ i với từ j")
print()

# In bảng scores
print("  RAW SCORES (trước softmax):")
header = "         " + "  ".join(f"{t:>6}" for t in demo_tokens)
print(header)
for i, tok in enumerate(demo_tokens):
    row = "  ".join(f"{scores_demo[i,j].item():6.2f}" for j in range(demo_seq))
    print(f"  {tok:6s} [ {row} ]")

print()
print("  ATTENTION WEIGHTS (sau softmax, mỗi hàng tổng = 1.0):")
print("  → đây chính là 'xác suất từ i chú ý vào từ j'")
print(header)
for i, tok in enumerate(demo_tokens):
    row = "  ".join(f"{attn_demo[i,j].item():6.3f}" for j in range(demo_seq))
    print(f"  {tok:6s} [ {row} ]")
    # Tìm từ được attend nhiều nhất
    max_j = attn_demo[i].argmax().item()
    print(f"         → '{tok}' chú ý nhiều nhất vào '{demo_tokens[max_j]}' ({attn_demo[i,max_j].item():.3f})")

print()
print("  KẾT LUẬN:")
print(f"  d_k = {demo_dk} chỉ là độ phong phú của vector mô tả khi tính dot product")
print(f"  Kết quả scores LUÔN là [{demo_seq}×{demo_seq}] = [seq_len × seq_len]")
print(f"  Không liên quan đến d_k — d_k chỉ ảnh hưởng ĐỘ CHÍNH XÁC của score")

DEMO: Cách Q @ K.T tạo ra ma trận 'từ nào attend từ nào'
  Q_demo shape : [4, 2]  ← mỗi hàng = vector Query của 1 từ
  K_demo shape : [4, 2]  ← mỗi hàng = vector Key của 1 từ

  Q_demo @ K_demo.T → scores shape : [4, 4]

  scores[i][j] = Q_i · K_j = dot product = 'mức độ match' của từ i với từ j

  RAW SCORES (trước softmax):
          [CLS]       I    love    deep
  [CLS]  [   0.71    0.71    0.00    0.71 ]
  I      [   0.71    0.71    0.00    0.71 ]
  love   [   0.00    0.00    0.71    0.71 ]
  deep   [   0.71    0.71    0.71    1.41 ]

  ATTENTION WEIGHTS (sau softmax, mỗi hàng tổng = 1.0):
  → đây chính là 'xác suất từ i chú ý vào từ j'
          [CLS]       I    love    deep
  [CLS]  [  0.286   0.286   0.141   0.286 ]
         → '[CLS]' chú ý nhiều nhất vào '[CLS]' (0.286)
  I      [  0.286   0.286   0.141   0.286 ]
         → 'I' chú ý nhiều nhất vào '[CLS]' (0.286)
  love   [  0.165   0.165   0.335   0.335 ]
         → 'love' chú ý nhiều nhất vào 'love' (0.335)
  deep   [  0.199

In [16]:
torch.manual_seed(0)
d_k = D_MODEL  # cho 1 head = d_model

# ── Ma trận trọng số W_Q, W_K, W_V ─────────────────────────────────────────────
W_Q = torch.randn(D_MODEL, d_k)   # [d_model, d_k]
W_K = torch.randn(D_MODEL, d_k)
W_V = torch.randn(D_MODEL, d_k)

print("=" * 60)
print("STEP 3: SELF-ATTENTION (1 head)")
print("=" * 60)
print(f"  d_k = {d_k}  (= d_model khi 1 head)")
print(f"  W_Q shape : {list(W_Q.shape)}")
print(f"  W_K shape : {list(W_K.shape)}")
print(f"  W_V shape : {list(W_V.shape)}")

# ── Tính Q, K, V ────────────────────────────────────────────────────────────────
# x: [1, seq_len, d_model]  →  squeeze → [seq_len, d_model]
x_sq = x.squeeze(0)          # [seq_len, d_model]

Q = x_sq @ W_Q               # [seq_len, d_k]
K = x_sq @ W_K               # [seq_len, d_k]
V = x_sq @ W_V               # [seq_len, d_k]

print()
print(f"  x_sq (input) shape : {list(x_sq.shape)}")
print(f"  Q shape            : {list(Q.shape)}")
print(f"  K shape            : {list(K.shape)}")
print(f"  V shape            : {list(V.shape)}")
print()
print("  Q (Query — 'từ này muốn tìm gì?'):")
for i, tok in enumerate(tokens):
    print(f"    [{i}] '{tok:10s}' → {Q[i].detach().numpy().round(3)}")

# ── Attention Score ──────────────────────────────────────────────────────────────
scores = Q @ K.T / np.sqrt(d_k)   # [seq_len, seq_len]

print()
print(f"  Attention scores (QK^T / sqrt({d_k})) shape : {list(scores.shape)}")
print("  Raw scores matrix (mỗi hàng = 1 từ, mỗi cột = score với từ kia):")
header = "       " + "  ".join(f"{t[:5]:>5}" for t in tokens)
print(header)
for i, tok in enumerate(tokens):
    row = "  ".join(f"{scores[i,j].item():5.2f}" for j in range(seq_len))
    print(f"  {tok[:6]:6s} [{row}]")

# ── Softmax → Attention Weights ──────────────────────────────────────────────────
attn_weights = F.softmax(scores, dim=-1)   # [seq_len, seq_len]

print()
print("  Attention weights sau softmax (mỗi hàng tổng = 1.0):")
print(header)
for i, tok in enumerate(tokens):
    row = "  ".join(f"{attn_weights[i,j].item():5.3f}" for j in range(seq_len))
    print(f"  {tok[:6]:6s} [{row}]")

# ── Weighted sum với V ───────────────────────────────────────────────────────────
attn_output = attn_weights @ V   # [seq_len, d_k]

print()
print(f"  Attention output (attn_weights @ V) shape : {list(attn_output.shape)}")
for i, tok in enumerate(tokens):
    print(f"  [{i}] '{tok:10s}' → {attn_output[i].detach().numpy().round(3)}")

STEP 3: SELF-ATTENTION (1 head)
  d_k = 8  (= d_model khi 1 head)
  W_Q shape : [8, 8]
  W_K shape : [8, 8]
  W_V shape : [8, 8]

  x_sq (input) shape : [9, 8]
  Q shape            : [9, 8]
  K shape            : [9, 8]
  V shape            : [9, 8]

  Q (Query — 'từ này muốn tìm gì?'):
    [0] '[CLS]     ' → [  4.872  -1.158   6.09   11.079 -11.46   -2.064   8.664   2.682]
    [1] 'i         ' → [ 11.804   6.437  -3.684  -1.886  -1.565  21.841 -10.157  -8.232]
    [2] 'love      ' → [ 5.069  7.849  2.431 18.2   -2.22  11.582  1.17  -7.218]
    [3] 'deep      ' → [ 7.349  5.623 -2.758 -2.313  6.185  3.628  2.148  4.357]
    [4] 'learning  ' → [  6.338   5.192  -4.791  -2.601   4.455   9.589 -10.176   0.77 ]
    [5] 'with      ' → [ 6.648 -9.741 -7.898 -2.506  5.729  5.633  1.022  7.168]
    [6] 'bert      ' → [ -2.721 -13.469   5.729  14.13    6.381  10.316  11.525 -11.489]
    [7] 'model     ' → [-12.737 -21.309   6.687  20.315   5.98   -3.383   8.619  -1.499]
    [8] '[SEP]     ' → [

---
## Multi-Head Attention
Chạy **NUM_HEADS** Self-Attention song song, mỗi head dùng d_k = d_model / num_heads.
Sau đó **concat** kết quả các head lại và nhân với W_O.

```
head_i = Attention(Q_i, K_i, V_i)
MultiHead = Concat(head_1, ..., head_h) × W_O
```

In [23]:
torch.manual_seed(1)
d_k_head = D_MODEL // NUM_HEADS   # chiều mỗi head
print('Chiều của mỗi head:', d_k_head)

print("=" * 60)
print("STEP 4: MULTI-HEAD ATTENTION")
print("=" * 60)
print(f"  num_heads  = {NUM_HEADS}")
print(f"  d_model    = {D_MODEL}")
print(f"  d_k / head = {d_k_head}  (mỗi head xử lý {d_k_head} chiều)")
print()

head_outputs = []

for h in range(NUM_HEADS):
    # Mỗi head có bộ W_Q, W_K, W_V riêng — shape [d_model, d_k_head]
    Wq_h = torch.randn(D_MODEL, d_k_head)
    Wk_h = torch.randn(D_MODEL, d_k_head)
    Wv_h = torch.randn(D_MODEL, d_k_head)

    Q_h = x_sq @ Wq_h                        # [seq_len, d_k_head]
    K_h = x_sq @ Wk_h
    V_h = x_sq @ Wv_h

    scores_h = Q_h @ K_h.T / np.sqrt(d_k_head)  # [seq_len, seq_len]
    attn_h   = F.softmax(scores_h, dim=-1)
    out_h    = attn_h @ V_h                      # [seq_len, d_k_head]

    head_outputs.append(out_h)

    print(f"  ── Head {h+1} ──────────────────────────────────────────")
    print(f"    Q_h shape        : {list(Q_h.shape)}")
    print(f"    scores_h shape   : {list(scores_h.shape)}")
    print(f"    attn_weights_h   : row sum = {attn_h.sum(dim=-1).detach().numpy().round(3)}  (phải = 1)")
    print(f"    head output shape: {list(out_h.shape)}")
    print(f"    head output[0] ('{tokens[0]}'): {out_h[0].detach().numpy().round(3)}")
    print()

# ── Concat tất cả heads ──────────────────────────────────────────────────────────
multi_head_concat = torch.cat(head_outputs, dim=-1)  # [seq_len, d_model]
print(f"  Concat {NUM_HEADS} heads → shape: {list(multi_head_concat.shape)}")
print(f"    (seq_len={seq_len}, {NUM_HEADS} heads × d_k={d_k_head} = d_model={D_MODEL})")
print(f"  multi_head_concat[0] ('{tokens[0]}'): {multi_head_concat[0].detach().numpy().round(3)}")

# ── W_O projection ────────────────────────────────────────────────────────────────
W_O = torch.randn(D_MODEL, D_MODEL)
mha_output = multi_head_concat @ W_O   # [seq_len, d_model]

print()
print(f"  W_O shape       : {list(W_O.shape)}")
print(f"  MHA output shape: {list(mha_output.shape)}")
print()
print("  MHA output (mỗi từ sau Multi-Head Attention):")
for i, tok in enumerate(tokens):
    print(f"  [{i}] '{tok:10s}' → {mha_output[i].detach().numpy().round(3)}")

Chiều của mỗi head: 4
STEP 4: MULTI-HEAD ATTENTION
  num_heads  = 2
  d_model    = 8
  d_k / head = 4  (mỗi head xử lý 4 chiều)

  ── Head 1 ──────────────────────────────────────────
    Q_h shape        : [9, 4]
    scores_h shape   : [9, 9]
    attn_weights_h   : row sum = [1. 1. 1. 1. 1. 1. 1. 1. 1.]  (phải = 1)
    head output shape: [9, 4]
    head output[0] ('[CLS]'): [-13.322  -8.224   0.566  -4.939]

  ── Head 2 ──────────────────────────────────────────
    Q_h shape        : [9, 4]
    scores_h shape   : [9, 9]
    attn_weights_h   : row sum = [1. 1. 1. 1. 1. 1. 1. 1. 1.]  (phải = 1)
    head output shape: [9, 4]
    head output[0] ('[CLS]'): [-8.163 -2.27   2.831  5.677]

  Concat 2 heads → shape: [9, 8]
    (seq_len=9, 2 heads × d_k=4 = d_model=8)
  multi_head_concat[0] ('[CLS]'): [-13.322  -8.224   0.566  -4.939  -8.163  -2.27    2.831   5.677]

  W_O shape       : [8, 8]
  MHA output shape: [9, 8]

  MHA output (mỗi từ sau Multi-Head Attention):
  [0] '[CLS]     ' → [  6

---
## Add & Norm (sau Attention)
**Residual connection** + Layer Normalization.
- **Add**: cộng input gốc vào output của Attention → tránh mất thông tin, giúp gradient chảy dễ hơn
- **Norm**: chuẩn hóa về mean=0, std=1

In [18]:
eps = 1e-6

# ── Residual Add ────────────────────────────────────────────────────────────────
add1 = x_sq + mha_output   # [seq_len, d_model]

print("=" * 60)
print("STEP 5: ADD & NORM (sau Attention)")
print("=" * 60)
print(f"  x_sq shape       : {list(x_sq.shape)}  ← input gốc (residual)")
print(f"  mha_output shape : {list(mha_output.shape)}  ← output Attention")
print(f"  add1 shape       : {list(add1.shape)}  = x_sq + mha_output")

# ── Layer Norm ──────────────────────────────────────────────────────────────────
mean1 = add1.mean(dim=-1, keepdim=True)   # [seq_len, 1]
var1  = add1.var(dim=-1, keepdim=True)    # [seq_len, 1]
norm1 = (add1 - mean1) / torch.sqrt(var1 + eps)  # [seq_len, d_model]

gamma1 = torch.ones(D_MODEL)
beta1  = torch.zeros(D_MODEL)
norm1_out = gamma1 * norm1 + beta1

print()
print("  Mean mỗi từ sau Add (trước norm):")
for i, tok in enumerate(tokens):
    print(f"    [{i}] '{tok:10s}'  mean={mean1[i,0].item():.4f}  var={var1[i,0].item():.4f}")

print()
print(f"  norm1_out shape : {list(norm1_out.shape)}")
print("  norm1_out (mỗi từ sau Add & Norm):")
for i, tok in enumerate(tokens):
    print(f"  [{i}] '{tok:10s}' → {norm1_out[i].detach().numpy().round(3)}")
    print(f"       mean={norm1_out[i].mean().item():.4f}  std={norm1_out[i].std().item():.4f}  (lý tưởng: ~0, ~1)")

STEP 5: ADD & NORM (sau Attention)
  x_sq shape       : [9, 8]  ← input gốc (residual)
  mha_output shape : [9, 8]  ← output Attention
  add1 shape       : [9, 8]  = x_sq + mha_output

  Mean mỗi từ sau Add (trước norm):
    [0] '[CLS]     '  mean=1.2623  var=299.9888
    [1] 'i         '  mean=2.1868  var=110.9103
    [2] 'love      '  mean=16.9586  var=675.6252
    [3] 'deep      '  mean=15.9077  var=671.3730
    [4] 'learning  '  mean=15.6768  var=665.4009
    [5] 'with      '  mean=15.6436  var=565.4569
    [6] 'bert      '  mean=16.6605  var=1251.6930
    [7] 'model     '  mean=22.9194  var=329.2539
    [8] '[SEP]     '  mean=22.4289  var=446.5081

  norm1_out shape : [9, 8]
  norm1_out (mỗi từ sau Add & Norm):
  [0] '[CLS]     ' → [ 0.196 -1.016 -1.397  0.948 -0.879  0.913  1.214  0.021]
       mean=-0.0000  std=1.0000  (lý tưởng: ~0, ~1)
  [1] 'i         ' → [-1.658 -0.45   0.427 -0.325 -0.625  1.235  1.358  0.038]
       mean=-0.0000  std=1.0000  (lý tưởng: ~0, ~1)
  [2] 'love 

---
## Feed Forward Network (FFN)
Hai lớp Linear với activation GELU ở giữa:

$$FFN(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$$

- **d_ff** thường = 4 × d_model trong BERT gốc
- Mỗi từ được xử lý **độc lập** qua FFN

In [19]:
torch.manual_seed(2)

# ── Trọng số FFN ────────────────────────────────────────────────────────────────
W1 = torch.randn(D_MODEL, D_FF)   # [d_model, d_ff]
b1 = torch.zeros(D_FF)
W2 = torch.randn(D_FF, D_MODEL)   # [d_ff, d_model]
b2 = torch.zeros(D_MODEL)

print("=" * 60)
print("STEP 6: FEED FORWARD NETWORK")
print("=" * 60)
print(f"  W1 shape : {list(W1.shape)}  [d_model={D_MODEL} → d_ff={D_FF}]")
print(f"  W2 shape : {list(W2.shape)}  [d_ff={D_FF} → d_model={D_MODEL}]")
print(f"  Input    : norm1_out shape = {list(norm1_out.shape)}")

# ── Forward pass FFN ─────────────────────────────────────────────────────────────
hidden = F.gelu(norm1_out @ W1 + b1)   # [seq_len, d_ff]
ff_out = hidden @ W2 + b2              # [seq_len, d_model]

print()
print(f"  Sau Linear1 + GELU : hidden shape = {list(hidden.shape)}")
print("  hidden[0] (CLS token, 4 giá trị đầu):", hidden[0, :4].detach().numpy().round(3))
print()
print(f"  Sau Linear2        : ff_out shape = {list(ff_out.shape)}")
print()
print("  FF output mỗi từ:")
for i, tok in enumerate(tokens):
    print(f"  [{i}] '{tok:10s}' → {ff_out[i].detach().numpy().round(3)}")

# ── Add & Norm lần 2 ─────────────────────────────────────────────────────────────
add2  = norm1_out + ff_out
mean2 = add2.mean(dim=-1, keepdim=True)
var2  = add2.var(dim=-1, keepdim=True)
norm2 = (add2 - mean2) / torch.sqrt(var2 + eps)

gamma2 = torch.ones(D_MODEL)
beta2  = torch.zeros(D_MODEL)
norm2_out = gamma2 * norm2 + beta2   # [seq_len, d_model]

print()
print("=" * 60)
print("STEP 6b: ADD & NORM (sau FFN)")
print("=" * 60)
print(f"  norm2_out shape : {list(norm2_out.shape)}")
print()
print("  Encoder Block output (1 layer):")
for i, tok in enumerate(tokens):
    print(f"  [{i}] '{tok:10s}' → {norm2_out[i].detach().numpy().round(3)}")

STEP 6: FEED FORWARD NETWORK
  W1 shape : [8, 16]  [d_model=8 → d_ff=16]
  W2 shape : [16, 8]  [d_ff=16 → d_model=8]
  Input    : norm1_out shape = [9, 8]

  Sau Linear1 + GELU : hidden shape = [9, 16]
  hidden[0] (CLS token, 4 giá trị đầu): [ 1.223 -0.114  0.859  3.12 ]

  Sau Linear2        : ff_out shape = [9, 8]

  FF output mỗi từ:
  [0] '[CLS]     ' → [ 2.668  6.472  5.335 -0.227 12.742 -9.197  1.115 -7.083]
  [1] 'i         ' → [ 10.721   2.386   1.69    5.322  20.337 -10.91    4.474  -6.819]
  [2] 'love      ' → [  2.741   8.15    5.41    3.44   22.08  -10.206   3.085  -7.596]
  [3] 'deep      ' → [  3.081   8.337   5.678   2.605  20.904 -10.597   1.654  -7.606]
  [4] 'learning  ' → [  3.282   8.105   4.835   3.109  21.069 -10.346   2.055  -7.992]
  [5] 'with      ' → [ 2.196  9.187  5.21   2.452 20.776 -9.554  1.775 -7.584]
  [6] 'bert      ' → [ 2.391  7.804  5.979  3.135 24.286 -8.274  2.857 -5.359]
  [7] 'model     ' → [ 2.351  5.771  5.732  5.794 21.174 -8.122  9.306 -4.58

---
## Encoder Block hoàn chỉnh (lớp class)
Đóng gói toàn bộ vào class `EncoderBlock` và chạy **NUM_LAYERS** lần (Nx).

In [20]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k     = d_model // num_heads
        self.h       = num_heads
        self.W_Q     = nn.Linear(d_model, d_model, bias=False)
        self.W_K     = nn.Linear(d_model, d_model, bias=False)
        self.W_V     = nn.Linear(d_model, d_model, bias=False)
        self.W_O     = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, verbose=False):
        B, T, C = x.shape   # batch, seq_len, d_model
        Q = self.W_Q(x).view(B, T, self.h, self.d_k).transpose(1, 2)  # [B,h,T,d_k]
        K = self.W_K(x).view(B, T, self.h, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.h, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / np.sqrt(self.d_k)  # [B,h,T,T]
        attn   = F.softmax(scores, dim=-1)
        out    = attn @ V                                       # [B,h,T,d_k]
        out    = out.transpose(1, 2).contiguous().view(B, T, C) # [B,T,d_model]
        out    = self.W_O(out)

        if verbose:
            print(f"    MHA: Q={list(Q.shape)} K={list(K.shape)} V={list(V.shape)}")
            print(f"    MHA: scores={list(scores.shape)} attn={list(attn.shape)}")
            print(f"    MHA: output={list(out.shape)}")
        return out, attn


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x, verbose=False):
        h = F.gelu(self.fc1(x))
        o = self.fc2(h)
        if verbose:
            print(f"    FFN: input={list(x.shape)} hidden={list(h.shape)} output={list(o.shape)}")
        return o


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn   = MultiHeadAttention(d_model, num_heads)
        self.ff     = FeedForward(d_model, d_ff)
        self.norm1  = nn.LayerNorm(d_model)
        self.norm2  = nn.LayerNorm(d_model)

    def forward(self, x, layer_idx=0, verbose=False):
        if verbose:
            print(f"  [Encoder layer {layer_idx}] input shape : {list(x.shape)}")

        # Sub-layer 1: MHA + Add & Norm
        attn_out, attn_w = self.attn(x, verbose=verbose)
        x = self.norm1(x + attn_out)
        if verbose:
            print(f"  [Encoder layer {layer_idx}] sau Add&Norm1 : {list(x.shape)}")

        # Sub-layer 2: FFN + Add & Norm
        ff_out = self.ff(x, verbose=verbose)
        x = self.norm2(x + ff_out)
        if verbose:
            print(f"  [Encoder layer {layer_idx}] sau Add&Norm2 : {list(x.shape)}")

        return x, attn_w


# ── Chạy toàn bộ encoder ─────────────────────────────────────────────────────────
torch.manual_seed(42)
encoder_layers = nn.ModuleList([EncoderBlock(D_MODEL, NUM_HEADS, D_FF) for _ in range(NUM_LAYERS)])

# Input: x [1, seq_len, d_model]
enc_input = x.clone()   # từ bước 2

print("=" * 60)
print(f"STEP 7: ENCODER — {NUM_LAYERS} LAYERS (Nx)")
print("=" * 60)
print(f"  enc_input shape : {list(enc_input.shape)}")
print()

enc_out = enc_input
all_attn = []
for i, layer in enumerate(encoder_layers):
    enc_out, attn_w = layer(enc_out, layer_idx=i+1, verbose=True)
    all_attn.append(attn_w)
    print()

print("=" * 60)
print("ENCODER OUTPUT (sau tất cả layers)")
print("=" * 60)
print(f"  enc_out shape : {list(enc_out.shape)}")
print()
for i, tok in enumerate(tokens):
    print(f"  [{i}] '{tok:10s}' → {enc_out[0, i].detach().numpy().round(3)}")

STEP 7: ENCODER — 2 LAYERS (Nx)
  enc_input shape : [1, 9, 8]

  [Encoder layer 1] input shape : [1, 9, 8]
    MHA: Q=[1, 2, 9, 4] K=[1, 2, 9, 4] V=[1, 2, 9, 4]
    MHA: scores=[1, 2, 9, 9] attn=[1, 2, 9, 9]
    MHA: output=[1, 9, 8]
  [Encoder layer 1] sau Add&Norm1 : [1, 9, 8]
    FFN: input=[1, 9, 8] hidden=[1, 9, 16] output=[1, 9, 8]
  [Encoder layer 1] sau Add&Norm2 : [1, 9, 8]

  [Encoder layer 2] input shape : [1, 9, 8]
    MHA: Q=[1, 2, 9, 4] K=[1, 2, 9, 4] V=[1, 2, 9, 4]
    MHA: scores=[1, 2, 9, 9] attn=[1, 2, 9, 9]
    MHA: output=[1, 9, 8]
  [Encoder layer 2] sau Add&Norm1 : [1, 9, 8]
    FFN: input=[1, 9, 8] hidden=[1, 9, 16] output=[1, 9, 8]
  [Encoder layer 2] sau Add&Norm2 : [1, 9, 8]

ENCODER OUTPUT (sau tất cả layers)
  enc_out shape : [1, 9, 8]

  [0] '[CLS]     ' → [-1.004  1.929 -0.111 -1.151 -0.579  0.2   -0.448  1.165]
  [1] 'i         ' → [-1.133  0.872 -0.845  0.243 -0.875  2.083 -0.268 -0.076]
  [2] 'love      ' → [-1.364 -0.281 -0.759  0.743 -0.662  1.755 -0.

---
## [CLS] token → Classifier
BERT dùng vector **[CLS]** (token đầu tiên) làm đại diện toàn câu → đưa qua Linear + Softmax để phân loại.

In [ ]:
NUM_CLASSES = 3  # ví dụ: positive / neutral / negative

torch.manual_seed(5)
classifier = nn.Linear(D_MODEL, NUM_CLASSES)

# ── Lấy [CLS] token (index 0) ────────────────────────────────────────────────────
cls_vector = enc_out[:, 0, :]   # [1, d_model]

print("=" * 60)
print("STEP 8: OUTPUT — [CLS] → CLASSIFIER")
print("=" * 60)
print(f"  enc_out shape   : {list(enc_out.shape)}  [batch, seq_len, d_model]")
print(f"  cls_vector shape: {list(cls_vector.shape)}  ← chỉ lấy token [CLS] (index 0)")
print(f"  cls_vector      : {cls_vector[0].detach().numpy().round(3)}")

# ── Linear ────────────────────────────────────────────────────────────────────────
logits = classifier(cls_vector)   # [1, num_classes]

print()
print(f"  Linear: {D_MODEL} → {NUM_CLASSES} classes")
print(f"  logits shape : {list(logits.shape)}")
print(f"  logits (raw) : {logits[0].detach().numpy().round(3)}")

# ── Softmax → Probabilities ───────────────────────────────────────────────────────
probs = F.softmax(logits, dim=-1)

classes = ["negative", "neutral", "positive"]
print()
print("  Output Probabilities:")
for cls_name, prob in zip(classes, probs[0]):
    bar = "█" * int(prob.item() * 30)
    print(f"    {cls_name:10s}: {prob.item():.4f}  {bar}")
print()
pred = probs.argmax(dim=-1).item()
print(f"  Prediction: '{classes[pred]}'  (class {pred})")

STEP 8: OUTPUT — [CLS] → CLASSIFIER
  enc_out shape   : [1, 9, 8]  [batch, seq_len, d_model]
  cls_vector shape: [1, 8]  ← chỉ lấy token [CLS] (index 0)
  cls_vector      : [-1.004  1.929 -0.111 -1.151 -0.579  0.2   -0.448  1.165]

  Linear: 8 → 3 classes
  logits shape : [1, 3]
  logits (raw) : [-0.625  0.849 -0.147]

  Output Probabilities:
    negative  : 0.1433  ████
    neutral   : 0.6257  ██████████████████
    positive  : 0.2310  ██████

  Prediction: 'neutral'  (class 1)


---
## So sánh với PyTorch Built-in
Dùng `nn.TransformerEncoder` để verify output có cùng cấu trúc.

In [22]:
torch.manual_seed(42)

# ── PyTorch built-in Encoder ──────────────────────────────────────────────────────
encoder_layer_pt  = nn.TransformerEncoderLayer(
    d_model   = D_MODEL,
    nhead     = NUM_HEADS,
    dim_feedforward = D_FF,
    dropout   = 0.0,
    activation = 'gelu',
    batch_first = True  # input shape [batch, seq, d_model]
)
transformer_encoder_pt = nn.TransformerEncoder(encoder_layer_pt, num_layers=NUM_LAYERS)

pt_out = transformer_encoder_pt(x)   # x: [1, seq_len, d_model]

print("=" * 60)
print("STEP 9: SO SÁNH VỚI PYTORCH BUILT-IN")
print("=" * 60)
print()
print(f"  Manual EncoderBlock output shape   : {list(enc_out.shape)}")
print(f"  PyTorch TransformerEncoder output  : {list(pt_out.shape)}")
print()
print("  ✓ Cùng shape → kiến trúc manual và built-in là tương đương")
print()
print("  (Giá trị số khác nhau vì trọng số random khác nhau,")
print("   nhưng cấu trúc pipeline hoàn toàn giống nhau)")
print()
print("  Manual output [CLS]  :", enc_out[0, 0].detach().numpy().round(3))
print("  PyTorch output [CLS] :", pt_out[0, 0].detach().numpy().round(3))

print()
print("=" * 60)
print("TÓM TẮT PIPELINE BERT")
print("=" * 60)
steps = [
    ("Input Text",                f"str"),
    ("Tokenize",                  f"[seq_len] = [{seq_len}] token IDs"),
    ("Word Embedding",            f"[{seq_len}, {D_MODEL}]"),
    ("+ Positional Encoding",     f"[{seq_len}, {D_MODEL}]"),
    ("unsqueeze (add batch dim)", f"[1, {seq_len}, {D_MODEL}]"),
    (f"Encoder ×{NUM_LAYERS}",   f"[1, {seq_len}, {D_MODEL}]"),
    ("CLS token",                 f"[1, {D_MODEL}]"),
    ("Linear Classifier",         f"[1, {NUM_CLASSES}]"),
    ("Softmax",                   f"[1, {NUM_CLASSES}]  ← probabilities"),
]
for name, shape in steps:
    print(f"  {name:30s} → {shape}")

STEP 9: SO SÁNH VỚI PYTORCH BUILT-IN

  Manual EncoderBlock output shape   : [1, 9, 8]
  PyTorch TransformerEncoder output  : [1, 9, 8]

  ✓ Cùng shape → kiến trúc manual và built-in là tương đương

  (Giá trị số khác nhau vì trọng số random khác nhau,
   nhưng cấu trúc pipeline hoàn toàn giống nhau)

  Manual output [CLS]  : [-1.004  1.929 -0.111 -1.151 -0.579  0.2   -0.448  1.165]
  PyTorch output [CLS] : [-0.26   1.991 -0.852 -0.314 -0.654  0.503 -1.298  0.884]

TÓM TẮT PIPELINE BERT
  Input Text                     → str
  Tokenize                       → [seq_len] = [9] token IDs
  Word Embedding                 → [9, 8]
  + Positional Encoding          → [9, 8]
  unsqueeze (add batch dim)      → [1, 9, 8]
  Encoder ×2                     → [1, 9, 8]
  CLS token                      → [1, 8]
  Linear Classifier              → [1, 3]
  Softmax                        → [1, 3]  ← probabilities
